# Klassifikation - Mindestanforderungen 6

#### Führen Sie mit dem Algorithmus Ihrer Wahl eine Klassifikationsaufgabe auf Ihren Daten durch.
- Durchführung Klassifikation mit XGBoost (XGBClassifier)
- Zielvariable: `Employment`
- Klassen:
    - `employed`
    - `independent contractor, freelancer, or self-employed`
    - `student`
    - `not employed`
- da `Employment` im One-Hot-Encoded Datensatz als `Employment_*` vorliegt, wurde Zielvariable pro Zeile wieder als einzelnes Label rekonstruiert und unbrauchbare Klassen wie `nan` und `i prefer not to say` entfernt
---
#### Teilen Sie dazu zunächst die Daten auf, um Overfitting beim Trainieren des Algorithmus und bei der Parameterauswahl zu vermeiden. Erklären Sie die gewählte Strategie und die Größenverhältnisse.
- **Gewählte Strategie**
  - **Stufe 1 (Modellwahl/Parametertuning)**
    - Parameter ausschließlich auf den Trainingsdaten optimiert –> dafür **3-fache Cross-Validation (cv=3)** im Trainingssplit
  - **Stufe 2 (Finale Bewertung)**
    - nachdem besten Parameter feststehen, wird finales Modell auf separaten Testsplit bewertet
  - durch `stratify=y` wird die Klassenverteilung in den Splits stabil gehalten

- **Konkrete Umsetzung im Code**
  - `train_test_split(..., test_size=0.2, stratify=y, random_state=42)` → **80% Train / 20% Test**
  - `GridSearchCV(..., cv=3)` → Cross-Validation innerhalb der **80% Trainingsdaten**

- **Größenverhältnisse:**
  - **Trainingsdaten/Testdaten:** 80% / 20%
---
#### Wählen Sie geeignete Features aus und setzen Sie die Parameter des Algorithmus. Beschreiben Sie das gewälhte Vorgehen für die Auswahl der Features und Parameter. Berichten Sie den Parameterraum und die final gewählten Parameter. Geben Sie die Performanz auf den Trainingsdaten (bzw. Entwicklungsdaten, falls verwendet) an.

- **Features**
    - Datengrundlage ist der One-Hot-Encodede Datensatz
    - `bool`-Werte wurden in 0/1 umgewandelt
    - entfernte Spalten
        - alle `Employment_*`-Spalten, da sonst schummeln
        - IDs und weitere nicht informative Variablen
        - `ConvertedCompTotal` - hatte viele Missing Values und keinen Fokus dieses Modells
    - Fehlende Werte in numerischen Features werden mit Median-Imputation aufgefüllt
- **Vorgehen**
    - Pipeline mit `ColumnTransformer` (MedianImputation) und XGBClassifier
    - Hyperparameter-Optimierung mit `GridSearchCV` (`cv=3`) auf Trainingssplit
    - Optimierungsmaß: macro-F1, da jede Klasse gleich gewichtet wird un Klassen unbalanciert sind
- **Parameterraum (GridSearch) - XGBoost (`XGBClassifier`)**
    - `n_estimators`: 300, 500
    - `max_depth`: 4, 6
    - `learning_rate`: 0.05, 0.1
    - `reg_lambda`: 1.0, 2.0
    - `subsample`: 0.8
    - `colsample_bytree`: 0.8
- **Finale Parameter**
    - `n_estimators=300`
    - `max_depth=4`
    - `learning_rate=0.05`
    - `reg_lambda=1.0`
    - `subsample=0.8`
    - `colsample_bytree=0.8`
- **Bestes Ergebnis aus Trainingsdaten**
    - 0.7696
---
#### Evaluieren Sie die Klassifikation auf den ungesehenen Testdaten. Betrachten Sie Precision und Recall sowie den F-Wert. Welches Maß ist für Ihre Anwendung wichtiger? Bewerten Sie Ihr Ergebnis. Ist es in der Praxis voraussichtlich zufriedenstellend?

- **Test-Performance**
    -   Accuracy: 0.95
    -   weighted: F1 0.94
    -   macro F1: 0.81
- **Klassenweise (Precision / Recall / F1)**
    - `employed`: 0.96 / 0.99 / 0.97
    - `independent contractor, freelancer, or self-employed`: 0.92 / 0.74 / 0.82
    - `not employed`: 0.86 / 0.71 / 0.78
    - `student`: 0.72 / 0.59 / 0.65
- **Interpretation**
    - meisten Fehler entstehen, indem Minderheitsklassen als `employed` vorhergesagt werden, da die Mehrheitsklasse dominiert
    - trotzdem werden Minderheitsklassen insgesamt gut erkannt
- **Wichtigstes Maß**
    - macro-F1 ist für Bewertung am wichtigsten, weil Klassenverteilung stark unausgewogen ist und Accuracy/weighted F1 vor allem durch `employed` geprägt werden
- **Praxisbewertung**
    - insgesamt zufriedenstellendes Ergebnis
    - gute Erkennung der Minderheitsklassen
    - für Anwendungen, bei denen Minderheitsklassen besonders kritisch sind, könnten zusätzliche Maßnahmen den Recall für `student` verbessern, sind aber für unser Ergebnis nicht zwingend notwendig


In [19]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier


- Einlesen One-Hot-Encodeder Datensatz
- Umwandlung von `bool`-Spalten in 0/1 (`int`), damit Features numerisch sind

In [20]:
df = pd.read_csv("One-Hot-Encoded-final.csv")

for c in df.select_dtypes(include=["bool"]).columns:
    df[c] = df[c].astype(int)

#### Zielvariable `Employment` aus One-Hot-Spalten rekonstruieren
- da Variable in One-Hot-Datensatz zu `Employment_*` encoded wurde, wird Zielvariable wieder als einzelnes Label pro Zeile gebaut
    - `emp_cols`: zeigt alle Spalten, die zu `Employment_*` gehören
    - `y_str`: Label wedern zu Strings, wie `employed` oder `student`
    - Unbrauchbare Klassen `nan`, `i prefer not to say` und `other` werden entfernt

In [21]:
emp_cols = [c for c in df.columns if c.startswith("Employment_")]
if not emp_cols:
    raise ValueError("Keine Employment_* Spalten gefunden.")

emp_mat = df[emp_cols].copy()

y_str = emp_mat.idxmax(axis=1).str.replace("Employment_", "", regex=False)

drop_classes = {"nan", "i prefer not to say", "Other"}
mask = ~y_str.isin(drop_classes)

df = df.loc[mask].copy()
y_str = y_str.loc[mask].copy()

print("Klassenverteilung:\n", y_str.value_counts())

Klassenverteilung:
 employed                                                15779
independent contractor, freelancer, or self-employed     2118
student                                                   487
not employed                                              210
Name: count, dtype: int64


#### Feature-Matrix `X` bestimmen
- alle `Employment_*`-Spalten (`emp_cols`) werden aus Features entfenrt, da das Modell sondt die Lösung direkt "sehen" würde
- zudem werden Spalten entfernt, die keinen inhaltlichen Mehrwert bieten
- am Ende werden nur numerische Spalten behalten, da dies für XGBoost relevant und notwendig ist

In [22]:
X = df.drop(columns=emp_cols + ["ResponseId", "AgeNum", "ConvertedCompTotal"], errors="ignore")

obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    print("Entferne verbliebene object-Spalten:", obj_cols)
    X = X.drop(columns=obj_cols)

# nur numerische Features
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
X = X[num_cols].copy()

print("X shape:", X.shape)

X shape: (18594, 378)


#### Zielvariable encoden
- XGBoost erwartet Klassen (0, 1,...,n)
- `LabelEncoder` wandelt String-Klassen (`y_str`) in Integer-Klassen um
- `class_names` wird gespeichert, um im späteren Report wieder lesbare Labels zu haben

In [23]:
le = LabelEncoder()
y = le.fit_transform(y_str)
class_names = le.classes_

#### Train/Test Split
- Aufteilen der Daten in 80% Training und 20% Test
- `stratify=y` sorgt für ungefähre Gleichverteilung der Klassen in Test und Trainingsdaten

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#### Preprocessing
- alle Features liegen numerish vor
- fehlende Werte werden mit Median aufgefüllt
-

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), num_cols)
    ],
    remainder="drop"
)

#### Modellpipeline und Suchraum definieren
- Pipeline kombiniert preprocessing und XGBoos Klassifikator
- Parameterraum ist kompakt gehalten:
    - `n_estimators`: Anzahl Bäume
    - `max_depth`: Baumtiefe (Modellkomplexität)
    - `learning_rate`: Schrittweite
    - `subsample`, colsample_bytree: Stochastik/Regularisierung gegen Overfitting
    - `reg_lambda`: L2-Regularisierung

In [26]:
xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42
)

pipe = Pipeline([
    ("preprocessing", preprocessor),
    ("clf", xgb)
])

param_grid = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [4, 6],
    "clf__learning_rate": [0.05, 0.1],
    "clf__subsample": [0.8],
    "clf__colsample_bytree": [0.8],
    "clf__reg_lambda": [1.0, 2.0],
}

- `GridSearchCV` testet alle Kombinationen aus `param_grid`
- Bewertung erfolgt über `f1_macro`, da jede Klasse gleich wichtig ist, trotz des Klassenunterschieds
- 3-fahe Cross-Validation auf dem Trainingssplit (`cv=3`)
- Testsplit bleibt unangetastet und dient später als faire Bewertung

In [27]:
grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\nBest CV macro-F1:", grid.best_score_)
print("Best params:", grid.best_params_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits

Best CV macro-F1: 0.7737493446146241
Best params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.05, 'clf__max_depth': 4, 'clf__n_estimators': 300, 'clf__reg_lambda': 1.0, 'clf__subsample': 0.8}


- bestes Modell aus GridSearch wird auf Testdaten evaluiert
- für Ausgabe werden integer-Klassen wieder in lesbare String-Labels gewandelt

In [28]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

y_test_lbl = le.inverse_transform(y_test)
y_pred_lbl = le.inverse_transform(y_pred)

print("\nTest report:\n", classification_report(y_test_lbl, y_pred_lbl, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test_lbl, y_pred_lbl, labels=class_names))


Test report:
                                                       precision    recall  f1-score   support

                                            employed       0.96      0.99      0.97      3156
independent contractor, freelancer, or self-employed       0.93      0.74      0.83       424
                                        not employed       0.83      0.71      0.77        42
                                             student       0.71      0.56      0.62        97

                                            accuracy                           0.95      3719
                                           macro avg       0.86      0.75      0.80      3719
                                        weighted avg       0.95      0.95      0.94      3719

Confusion matrix:
 [[3125   18    0   13]
 [ 105  315    0    4]
 [   2    5   30    5]
 [  36    1    6   54]]


In [29]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))

Classification Report (Train):
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     12623
           1       0.95      0.76      0.84      1694
           2       1.00      0.79      0.88       168
           3       0.96      0.70      0.81       390

    accuracy                           0.96     14875
   macro avg       0.97      0.81      0.88     14875
weighted avg       0.96      0.96      0.96     14875

